# 01 · Benchmark data, task selection, contamination ledger
Primary benchmark: **TIME** (ICML 2026) — fresh datasets built to avoid leakage into foundation-model pretraining, 98 forecasting tasks with
short/medium/long terms. Everything below is deterministic (seed in `configs/protocol.yaml`) so the professor can re-run it.

In [ ]:
import os, sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, yaml
pd.set_option("display.width", 160, "display.max_columns", 30)

In [ ]:
from tscompare.tasks import TimeBenchSource, make_synthetic_task
from tscompare.selection import select_core_tasks
P = yaml.safe_load(open("configs/protocol.yaml")); caps = P["data_caps"]
HAVE_TIME = bool(os.environ.get("TIME_STORAGE"))
print("TIME available:", HAVE_TIME)

## 1 · Enumerate tasks and pick the core subset

In [ ]:
if HAVE_TIME:
    src = TimeBenchSource(os.environ["TIME_STORAGE"])
    all_tasks = src.list_tasks(); display(all_tasks.head())
    print(len(all_tasks), "tasks;", all_tasks.dataset.nunique(), "datasets")
else:  # demo so the notebook runs end-to-end without the data
    all_tasks = pd.DataFrame([dict(name=f"demo{i}/{f}", dataset=f"demo{i}", freq=f, term=t, domain=d, prediction_length=h)
        for i, (f, d) in enumerate([("H","energy"),("D","sales"),("5T","cloud"),("H","transport"),("ME","econ"),("15T","energy")])
        for t, h in [("short",24),("medium",48)]])
    display(all_tasks.head())

In [ ]:
N_CORE = 20                      # raise when you have more compute; 98 = everything (zero-shot only for the full set)
core = select_core_tasks(all_tasks, n=N_CORE, seed=caps["seed"])
core[["name", "term"]].to_dict("records")
yaml.safe_dump({"tasks": core[["name", "term"]].to_dict("records")}, open("configs/tasks_core.yaml", "w"))
core.groupby(["domain", "term"]).size().unstack(fill_value=0)

Check the coverage table: every domain and every term should appear. If a domain is missing, raise `N_CORE` or edit `configs/tasks_core.yaml` by hand
(commit the file — it *is* the protocol).

## 2 · Load one task and inspect it

In [ ]:
if HAVE_TIME:
    r = core.iloc[0]; task = src.load(r["name"], r["term"], max_instances=caps["max_instances_per_task"],
                                      max_train_series=caps["max_train_series"], max_context_store=caps["max_context_store"], seed=caps["seed"])
else:
    task = make_synthetic_task()
print(task.summary())
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 3)); c, y = task.contexts[0][-200:], task.targets[0]
ax.plot(range(len(c)), c, label="context"); ax.plot(range(len(c), len(c) + len(y)), y, label="target"); ax.legend(); plt.show()

### Leakage assertions (must pass — these protect the validity of the whole paper)

In [ ]:
if task.train_series is not None:
    # supervised methods only ever see train_series/val_*; test targets are never inside them
    max_train_end = max(len(s) for s in task.train_series)
    print("longest train series:", max_train_end, "| shortest eval context:", min(len(c) for c in task.contexts))
assert task.targets.shape == (task.n, task.horizon)
assert np.isfinite(task.mase_scale).mean() > 0.5, "too many series with undefined MASE scale"
print("OK")

## 3 · Dataset statistics table (goes into the paper's appendix)

In [ ]:
rows = []
for _, r in core.iterrows():
    try:
        t = src.load(r["name"], r["term"], max_instances=caps["max_instances_per_task"], max_train_series=caps["max_train_series"],
                     max_context_store=caps["max_context_store"], need_train=False) if HAVE_TIME else make_synthetic_task(seed=1)
        rows.append({**t.summary(), "domain": r["domain"]})
    except Exception as e:
        print("skip", r["name"], e)
stats = pd.DataFrame(rows); stats

In [ ]:
Path("paper/tables").mkdir(parents=True, exist_ok=True)
cols = [c for c in ["task", "domain", "freq", "term", "horizon", "season", "n_instances", "ctx_median"] if c in stats]
stats[cols].to_latex("paper/tables/datasets.tex", index=False, escape=True, longtable=False)

## 4 · Sanity check against the public leaderboard (do this once, it catches protocol bugs)
Run `seasonal_naive` through *our* harness and compare per-task MASE to the seasonal-naive numbers published in the TIME results repo
(`TIME-Output` on HF). Agreement within a few % ⇒ our windows, scaling and metric implementation match theirs. Large gaps ⇒ fix before running anything else.

In [ ]:
from tscompare import runner
from tscompare.models.registry import build
if HAVE_TIME:
    tasks = [src.load(r["name"], r["term"], max_instances=caps["max_instances_per_task"], need_train=False, seed=caps["seed"]) for _, r in core.iterrows()]
    runner.run(build("seasonal_naive"), tasks, out_dir="results", overwrite=True)
    # TODO: download TIME-Output, read seasonal_naive per-task MASE, merge on task id, print relative differences.
else:
    print("skipped (needs TIME data)")

## 5 · Contamination ledger
Fill `configs/contamination.csv` (declared pretraining corpora vs the dataset sources listed in the TIME paper appendix). Then mark tasks per model family
in `task.meta["contaminated"]`; the analysis notebook reports results **with and without** flagged pairs. LLM-based methods cannot be audited at series level — say so in Limitations.

In [ ]:
pd.read_csv("configs/contamination.csv")